# 🦷 Dental Aligner GRPO Training on Colab

Train a small LLM (Qwen2.5-0.5B) to plan orthodontic aligner stages using GRPO.

**Requirements:** GPU runtime (T4 is fine). Go to Runtime → Change runtime type → T4 GPU.

**Time:** ~30-60 min for 20 episodes on T4.

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clone the repo
!git clone https://github.com/mehular0ra/dental-aligner-claw4s.git
%cd dental-aligner-claw4s

In [ ]:
# Install dependencies
!pip install -q trl transformers accelerate peft wandb
!pip install -q fastapi uvicorn pydantic scipy numpy
# Install openenv (the environment framework)
!pip install -q openenv 2>/dev/null || echo 'openenv not on PyPI, using local'

## 2. Start the Dental Aligner Environment Server

In [ ]:
import subprocess, time, json, urllib.request

# Start server in background
server = subprocess.Popen(
    ['python', '-m', 'server.app'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(8)

# Health check
try:
    resp = json.loads(urllib.request.urlopen('http://localhost:7860/health').read())
    print(f'Server: {resp}')
except Exception as e:
    print(f'Server failed to start: {e}')
    print('Trying alternative start...')
    !python -m server.app &
    time.sleep(10)
    resp = json.loads(urllib.request.urlopen('http://localhost:7860/health').read())
    print(f'Server: {resp}')

In [ ]:
# Quick verification: run benchmark
!python benchmarks.py --quick

## 3. Login to Weights & Biases

In [ ]:
import wandb
wandb.login()  # Paste your API key from wandb.ai/authorize

## 4. Generate Training Dataset

In [ ]:
import json, math, urllib.request

SERVER = 'http://localhost:7860'

def post(endpoint, data):
    req = urllib.request.Request(
        f'{SERVER}{endpoint}',
        data=json.dumps(data).encode(),
        headers={'Content-Type': 'application/json'}
    )
    return json.loads(urllib.request.urlopen(req, timeout=30).read().decode(), strict=False)

def make_prompt(obs, stage=0):
    progress = obs.get('per_tooth_progress', [0]*28)
    mean_p = sum(progress) / max(len(progress), 1)
    min_p = min(progress) if progress else 0
    return f"""You are an orthodontic treatment planning AI. Plan aligner stage {stage+1}/24.

STATE: Stage {stage}/24, Progress: {mean_p:.0%}, Worst: {min_p:.0%}, Violations: {obs.get('cumulative_violations',0)}

RULES: Max 0.25mm/2deg per tooth per stage. Move incisors BEFORE molars.

Output JSON: {{"strategy": "...", "tooth_groups": [{{"teeth": [...], "fraction": 0.0-1.0}}]}}
Where fraction=interpolation toward target. System computes SE(3) poses via SLERP."""

# Generate prompts from different seeds and difficulties
N_EPISODES = 20  # Increase for better training
prompts = []

difficulties = [
    {'n_perturbed_teeth': 6, 'translation_magnitude': 2.0},
    {'n_perturbed_teeth': 12, 'translation_magnitude': 3.5},
    {'n_perturbed_teeth': 18, 'translation_magnitude': 5.0},
]

for i in range(N_EPISODES):
    seed = i * 7 + 42
    diff = difficulties[i % len(difficulties)]
    obs = post('/reset_stepwise', {
        'task_id': 'task_easy', 'seed': seed, 'source': 'synthetic',
        'episode_id': f'prompt_{i}', 'difficulty_params': diff
    })
    prompts.append({
        'prompt': make_prompt(obs),
        'seed': seed,
        'difficulty_params': diff,
    })

print(f'Generated {len(prompts)} training prompts')
print(f'Sample:\n{prompts[0]["prompt"][:300]}...')

## 5. Define Reward Functions

In [ ]:
def run_episode_from_completion(completion_text, seed, difficulty_params):
    """Parse LLM completion into a staging plan, run 24-step episode, return scores."""
    # Parse LLM output
    tooth_fractions = {}
    try:
        js = completion_text[completion_text.find('{'):completion_text.rfind('}')+1]
        plan = json.loads(js)
        for g in plan.get('tooth_groups', []):
            for tid in g.get('teeth', []):
                tooth_fractions[tid] = float(g.get('fraction', 0.5))
    except:
        pass

    TOOTH_IDS = [11,12,13,14,15,16,17,21,22,23,24,25,26,27,
                 31,32,33,34,35,36,37,41,42,43,44,45,46,47]

    eid = f'grpo_{seed}_{hash(completion_text) % 10000}'
    obs = post('/reset_stepwise', {
        'task_id': 'task_easy', 'seed': seed, 'source': 'synthetic',
        'episode_id': eid, 'difficulty_params': difficulty_params
    })
    init, tgt = obs['current_config'], obs['target_config']

    final = None
    for s in range(1, 25):
        alpha_default = s / 25.0
        poses = []
        for i in range(28):
            tid = TOOTH_IDS[i]
            frac = tooth_fractions.get(tid, alpha_default)
            frac = max(0.0, min(1.0, frac))
            q = [init[i][j]*(1-frac)+tgt[i][j]*frac for j in range(4)]
            qn = math.sqrt(sum(x*x for x in q))
            q = [x/max(qn,1e-10) for x in q]
            t = [init[i][4+j]*(1-frac)+tgt[i][4+j]*frac for j in range(3)]
            poses.append(q+t)
        final = post('/step_stepwise', {'episode_id': eid, 'poses': poses})

    return final

# Reward functions (decomposed for GRPO)
def reward_terminal(completions, prompts, **kwargs):
    """Terminal reward (accuracy + smoothness + compliance + staging)."""
    rewards = []
    for i, comp in enumerate(completions):
        try:
            idx = i % len(prompts)
            obs = run_episode_from_completion(
                comp[0]['content'] if isinstance(comp, list) else str(comp),
                prompts[idx]['seed'],
                prompts[idx]['difficulty_params']
            )
            rewards.append(float(obs.get('terminal_reward', 0.0)))
        except Exception as e:
            rewards.append(0.0)
    return rewards

def reward_occlusion(completions, prompts, **kwargs):
    """Andrews' Six Keys occlusion composite."""
    rewards = []
    for i, comp in enumerate(completions):
        try:
            idx = i % len(prompts)
            obs = run_episode_from_completion(
                comp[0]['content'] if isinstance(comp, list) else str(comp),
                prompts[idx]['seed'],
                prompts[idx]['difficulty_params']
            )
            bd = obs.get('reward_breakdown', {})
            rewards.append(float(bd.get('occlusion_composite', 0.0)))
        except:
            rewards.append(0.0)
    return rewards

print('Reward functions defined. Testing...')
test_comp = json.dumps({'strategy': 'test', 'tooth_groups': [{'teeth': [11,21], 'fraction': 0.5}]})
test_r = reward_terminal([test_comp], prompts[:1])
print(f'Test reward: {test_r}')

## 6. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Format prompts for TRL
train_dataset = []
for p in prompts:
    train_dataset.append({
        'prompt': [{'role': 'user', 'content': p['prompt']}],
    })

config = GRPOConfig(
    output_dir='./dental_grpo_output',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,
    num_train_epochs=1,
    save_steps=5,
    logging_steps=1,
    report_to='wandb',
    run_name='dental-grpo-qwen0.5b',
    bf16=torch.cuda.is_available(),
)

print(f'Training config: {config.per_device_train_batch_size} batch, {config.num_generations} gens, {config.num_train_epochs} epochs')
print(f'Total steps: ~{len(train_dataset) * config.num_train_epochs}')

In [ ]:
# Launch training
trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=[reward_terminal, reward_occlusion],
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print('Starting GRPO training...')
trainer.train()

In [ ]:
# Save trained model
trainer.save_model('./dental_grpo_output/final')
print('Model saved to ./dental_grpo_output/final')
print(f'wandb run: {wandb.run.get_url() if wandb.run else "not logged"}')

## 7. Evaluate Trained Model

In [ ]:
from transformers import pipeline

# Load trained model
generator = pipeline('text-generation', model='./dental_grpo_output/final', tokenizer=tokenizer, device=0)

# Generate plan for a test case
test_obs = post('/reset_stepwise', {'task_id': 'task_easy', 'seed': 999, 'source': 'synthetic', 'episode_id': 'eval_1'})
test_prompt = make_prompt(test_obs)

output = generator(test_prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
completion = output[0]['generated_text'][len(test_prompt):]
print(f'Model output:\n{completion[:500]}')

# Run episode with model's plan
result = run_episode_from_completion(completion, 999, {'n_perturbed_teeth': 10, 'translation_magnitude': 3.0})
print(f'\nTerminal reward: {result.get("terminal_reward", 0):.4f}')
print(f'Occlusion: {result.get("reward_breakdown", {}).get("occlusion_composite", 0):.4f}')

# Compare with SLERP baseline
print(f'\nSLERP baseline for same case: ~0.87')
print(f'Improvement: {(result.get("terminal_reward", 0) - 0.87) * 100:+.1f}%')

## 8. Download Results

Download the trained model and wandb logs to include in your CLAW4S submission.

In [ ]:
# Zip results for download
!zip -r dental_grpo_results.zip dental_grpo_output/final/ 2>/dev/null

from google.colab import files
files.download('dental_grpo_results.zip')
print('Download complete. Add the wandb link to your research note.')